In [1]:

import os


def repo_root():
    """Locate the repository root (the directory that contains data/)."""
    cur = os.getcwd()
    for _ in range(8):
        if os.path.isdir(os.path.join(cur, 'data')) and os.path.exists(
            os.path.join(cur, 'data', 'appointments.csv')
        ):
            return cur
        nxt = os.path.dirname(cur)
        if nxt == cur:
            break
        cur = nxt
    raise FileNotFoundError('Could not locate the repository data/ directory')


ROOT = repo_root()
OUTPUT_DIR = ROOT
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Repository root:', ROOT)
print('Output directory:', OUTPUT_DIR)


Repository root: /Users/harish/careloop-clinics-no-show-guard
Output directory: /Users/harish/careloop-clinics-no-show-guard


## Task 1.1 — CSV ingestion

Objective: ingest `appointments.csv` using only Python's `csv` module, type-validate each row, and separate accepted rows from rejected rows.

Validation rules:
- `age`, `fee_inr` and `prior_no_shows` must convert to `int`.
- `distance_km` must convert to `float`.
- Rows that fail any conversion are rejected.
- Impossible ages (e.g. negative or over 100) still pass this type-validation task; they are handled later in Task 1.3.

Rejected rows are saved to `output/rejected_rows.json`.

In [2]:
import csv
import json
import os

APPOINTMENTS_CSV = os.path.join(ROOT, 'data', 'appointments.csv')
print('Reading:', APPOINTMENTS_CSV)


def parse_row(row):
    """Convert the typed fields of one CSV row and return the dictionary."""
    row['age'] = int(row['age'])
    row['fee_inr'] = int(row['fee_inr'])
    row['prior_no_shows'] = int(row['prior_no_shows'])
    row['distance_km'] = float(row['distance_km'])
    return row


accepted = []
rejected = []

with open(APPOINTMENTS_CSV, newline='') as f:
    reader = csv.DictReader(f)
    headers = reader.fieldnames
    print('Headers:', headers)
    for row in reader:
        try:
            accepted.append(parse_row(dict(row)))
        except ValueError:
            rejected.append(dict(row))

print('Total rows:', len(accepted) + len(rejected))
print('Accepted:', len(accepted))
print('Rejected:', len(rejected))
print('Rejected with blank age:', sum(1 for r in rejected if str(r['age']).strip() == ''))
print('Rejected with blank distance_km:', sum(1 for r in rejected if str(r['distance_km']).strip() == ''))

REJECTED_JSON = os.path.join(ROOT, 'rejected_rows.json')
with open(REJECTED_JSON, 'w') as f:
    json.dump(rejected, f, indent=2)
print('Wrote', REJECTED_JSON)


Reading: /Users/harish/careloop-clinics-no-show-guard/data/appointments.csv
Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows: 1020
Accepted: 988
Rejected: 32
Rejected with blank age: 12
Rejected with blank distance_km: 20
Wrote /Users/harish/careloop-clinics-no-show-guard/rejected_rows.json


## Task 1.2 — Initial pandas cleaning

Objective: reload the **original** `appointments.csv` with pandas and clean it in a fixed order, printing the row count after each stage.

Cleaning order:
1. Remove exact duplicate rows.
2. Normalize `booking_channel` with `str.strip().str.title()`.
3. Convert `appt_date` with `pd.to_datetime(errors="coerce")`.
4. Convert `booked_on` with `pd.to_datetime()`.
5. Remove rows with missing `appt_date`.
6. Remove rows with missing `status`.

The `df` from this task is reused by Tasks 1.3–1.6.

In [3]:

import pandas as pd

df = pd.read_csv(os.path.join(ROOT, 'data', 'appointments.csv'))

print('Initial shape:', df.shape)
print(df.info())
print()
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Exact duplicate rows:', int(df.duplicated().sum()))
print('Raw booking_channel spellings:', df['booking_channel'].nunique())

# 1. Remove exact duplicate rows.
df = df.drop_duplicates()
print()
print('Rows after removing duplicates:', len(df))

# 2. Normalize booking_channel.
df['booking_channel'] = df['booking_channel'].str.strip().str.title()

# 3. Convert appt_date (unreadable values become NaT).
df['appt_date'] = pd.to_datetime(df['appt_date'], errors='coerce')
unreadable_dates = int(df['appt_date'].isnull().sum())
print('Unreadable appt_date values:', unreadable_dates)

# 4. Convert booked_on.
df['booked_on'] = pd.to_datetime(df['booked_on'])

# 5. Remove rows with missing appt_date.
df = df.dropna(subset=['appt_date'])
print('Rows after removing missing appt_date:', len(df))

# 6. Remove rows with missing status.
df = df.dropna(subset=['status'])
print('Rows after removing missing status:', len(df))

print()
print('Booking channel labels:', sorted(df['booking_channel'].unique().tolist()))
print('No-show appointments:', int((df['status'] == 'No-Show').sum()))


Initial shape: (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB
None

Missing values per column:
appointment_id      0
patient_id 

## Task 1.3 — Validity, missing values and outliers

Objective: handle invalid ages, missing values, and distance/fee outliers in the exact specified order, using statistics calculated from the data (no hardcoding).

Order:
1. Replace ages below 1 or above 100 with missing values.
2. Fill missing age with the median.
3. Round age and convert to integer.
4. Fill missing `distance_km` with the median.
5–7. Compute distance quartiles/IQR/fences and cap with `np.clip`.
8. Repeat the fencing/capping for `fee_inr`.
9–10. Round `distance_km` to one decimal and `fee_inr` to an integer.

In [4]:

import numpy as np

# 1. Replace ages below 1 or above 100 with missing values.
invalid_age_mask = (df['age'] < 1) | (df['age'] > 100)
invalid_age_count = int(invalid_age_mask.sum())
df.loc[invalid_age_mask, 'age'] = np.nan

print('Invalid age count:', invalid_age_count)
print('Missing ages after invalid-value detection:', int(df['age'].isnull().sum()))

# 2–3. Fill missing age with median, round, convert to integer.
median_age = df['age'].median()
df['age'] = df['age'].fillna(median_age).round().astype(int)
print('Median age:', median_age)

# 4. Fill missing distance_km with median.
median_distance = df['distance_km'].median()
df['distance_km'] = df['distance_km'].fillna(median_distance)
print('Median distance:', median_distance)

# 5–7. Distance quartiles, IQR, fences, cap with np.clip.
q1 = df['distance_km'].quantile(0.25)
q3 = df['distance_km'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
print('Distance Q1:', q1)
print('Distance Q3:', q3)
print('Distance IQR:', iqr)
print('Distance lower fence:', lower_fence)
print('Distance upper fence:', upper_fence)

distance_before = df['distance_km'].copy()
df['distance_km'] = np.clip(df['distance_km'], lower_fence, upper_fence)
print('Number of capped distance values:', int((distance_before != df['distance_km']).sum()))

# 8. Repeat for fee_inr.
q1f = df['fee_inr'].quantile(0.25)
q3f = df['fee_inr'].quantile(0.75)
iqrf = q3f - q1f
lower_fence_f = q1f - 1.5 * iqrf
upper_fence_f = q3f + 1.5 * iqrf
print('Fee Q1:', q1f)
print('Fee Q3:', q3f)
print('Fee IQR:', iqrf)
print('Fee lower fence:', lower_fence_f)
print('Fee upper fence:', upper_fence_f)

fee_before = df['fee_inr'].copy()
df['fee_inr'] = np.clip(df['fee_inr'], lower_fence_f, upper_fence_f)
print('Number of capped fee values:', int((fee_before != df['fee_inr']).sum()))

# 9–10. Round.
df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round().astype(int)
print('Maximum final fee:', int(df['fee_inr'].max()))


Invalid age count: 5
Missing ages after invalid-value detection: 17
Median age: 48.0
Median distance: 8.25
Distance Q1: 4.625
Distance Q3: 13.0
Distance IQR: 8.375
Distance lower fence: -7.9375
Distance upper fence: 25.5625
Number of capped distance values: 26
Fee Q1: 500.0
Fee Q3: 787.5
Fee IQR: 287.5
Fee lower fence: 68.75
Fee upper fence: 1218.75
Number of capped fee values: 3
Maximum final fee: 1219


## Task 1.4 — Feature engineering and NumPy

Objective: derive the `lead_days` and `no_show` fields, build the NumPy feature matrix `X`, and compute column statistics without loops.

- `lead_days = (appt_date - booked_on).dt.days`
- `no_show = 1` if `status == 'No-Show'`, else `0`
- `X = df[['age', 'distance_km', 'fee_inr', 'lead_days']].to_numpy()`
- Column means with `axis=0`, count/share of `lead_days > 14`, distance z-scores (population std), count of `|z| > 2`.

In [5]:

import numpy as np

df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days
df['no_show'] = (df['status'] == 'No-Show').astype(int)

X = df[['age', 'distance_km', 'fee_inr', 'lead_days']].to_numpy()

print('Column means (axis=0):', X.mean(axis=0))
print('Lead days range:', int(X[:, 3].min()), '-', int(X[:, 3].max()))
print('No-shows:', int(df['no_show'].sum()))
print('No-show rate: {:.1f}%'.format(df['no_show'].mean() * 100))

lead_gt_14_mask = X[:, 3] > 14
print('Lead days above 14:', int(lead_gt_14_mask.sum()))
print('Share of bookings with lead_days > 14: {:.1f}%'.format(lead_gt_14_mask.mean() * 100))

distance = X[:, 1]
z_scores = (distance - distance.mean()) / distance.std(ddof=0)
print('Distance z-score outliers (|z| > 2):', int((np.abs(z_scores) > 2).sum()))


Column means (axis=0):

 [ 48.17373737   9.50090909 635.71414141  11.81919192]
Lead days range: 1 - 45
No-shows: 218
No-show rate: 22.0%
Lead days above 14: 296
Share of bookings with lead_days > 14: 29.9%
Distance z-score outliers (|z| > 2): 53


## Task 1.5 — Open-Meteo enrichment

Objective: enrich the eight clinics with the current forecast temperature from Open-Meteo.

For each clinic, request `hourly=temperature_2m` with `forecast_days=1` using the original latitude/longitude, and store the first temperature value as `temp_c`.

- `requests.get` with `timeout=10`
- HTTP status validation (`raise_for_status`)
- JSON validation
- `try/except requests.exceptions.RequestException`
- One retry after a failed request
- On persistent failure: store `None`, print the clinic name and reason, continue

Temperatures are never invented.

In [6]:
import requests
import pandas as pd

clinics = pd.read_csv(os.path.join(ROOT, 'data', 'clinics.csv'))


def fetch_temperature(clinic, retries=1):
    """Return (temp_c, reason). Never invents a temperature."""
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude': clinic['latitude'],
        'longitude': clinic['longitude'],
        'hourly': 'temperature_2m',
        'forecast_days': 1,
    }
    for attempt in range(retries + 1):
        reason = ''
        try:
            response = requests.get(url, params=params, timeout=10)
            if response.status_code == 200:
                data = response.json()
                temps = data.get('hourly', {}).get('temperature_2m', [])
                if temps:
                    return float(temps[0]), ''
                reason = 'hourly temperature_2m missing'
            else:
                reason = f'HTTP status {response.status_code}'
        except requests.exceptions.RequestException as exc:
            reason = f'{type(exc).__name__}: {exc}'
    return None, reason


temperatures = []
for _, clinic in clinics.iterrows():
    temp, reason = fetch_temperature(clinic)
    if temp is None:
        print(clinic['clinic_name'], '-', reason, '- temp_c stored as None / SQL NULL')
    temperatures.append(temp)

clinics['temp_c'] = temperatures
CLINICS_ENRICHED = os.path.join(ROOT, 'clinics_enriched.csv')
clinics.to_csv(CLINICS_ENRICHED, index=False)
print()
print('Wrote', CLINICS_ENRICHED)
print(clinics[['clinic_id', 'clinic_name', 'city', 'temp_c']].to_string(index=False))


/Users/harish/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(



Wrote /Users/harish/careloop-clinics-no-show-guard/clinics_enriched.csv
clinic_id              clinic_name    city  temp_c
      C01    Kothrud Family Clinic    Pune    19.2
      C02     Hadapsar Care Centre    Pune    19.5
      C03   Sitabuldi Health Point  Nagpur    18.6
      C04     Malviya Nagar Clinic  Jaipur    14.4
      C05    Vaishali Nagar Clinic  Jaipur    14.4
      C06 Edappally Medical Centre   Kochi    27.9
      C07       Vijay Nagar Clinic  Indore    16.6
      C08   Gomti Nagar Care Point Lucknow    14.0


## Task 1.6 — Export clean data and SQL loading script

Objective: export the cleaned appointments and write `output/load_data.sql`.

- Sort by `appointment_id`.
- Format dates as `YYYY-MM-DD`.
- Export `appointments_clean.csv` with exactly 16 columns in the specified order.
- Verify `appointment_id` uniqueness and zero missing values across all 16 columns.
- `load_data.sql`: all eight clinic `INSERT`s first, then all appointment `INSERT`s, built with a `for` loop and f-strings. Missing temperature is SQL `NULL` without quotes.

In [7]:
import pandas as pd

df = df.sort_values('appointment_id').reset_index(drop=True)
df['booked_on'] = df['booked_on'].dt.strftime('%Y-%m-%d')
df['appt_date'] = df['appt_date'].dt.strftime('%Y-%m-%d')

COLUMNS = [
    'appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date',
    'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel',
    'visit_type', 'reminder_sent', 'prior_no_shows', 'status',
    'lead_days', 'no_show',
]

print('appointment_id unique:', bool(df['appointment_id'].is_unique))
missing_counts = df[COLUMNS].isnull().sum()
print('Total missing values across the 16 columns:', int(missing_counts.sum()))
print(missing_counts.to_string())

CLEAN_CSV = os.path.join(ROOT, 'appointments_clean.csv')
df[COLUMNS].to_csv(CLEAN_CSV, index=False)
print('Wrote', CLEAN_CSV, '| rows:', len(df), '| columns:', len(COLUMNS))

clinics_df = pd.read_csv(CLINICS_ENRICHED)

lines = []
lines.append('-- Clinic records (8)')
for _, c in clinics_df.iterrows():
    temp = 'NULL' if pd.isna(c['temp_c']) else str(c['temp_c'])
    lines.append(
        f"INSERT INTO clinics VALUES ('{c['clinic_id']}','{c['clinic_name']}',"
        f"'{c['city']}','{c['tier']}',{c['latitude']},{c['longitude']},{temp});"
    )

lines.append('')
lines.append('-- Appointment records (990)')
for _, a in df.iterrows():
    lines.append(
        f"INSERT INTO appointments VALUES ('{a['appointment_id']}','{a['patient_id']}',"
        f"'{a['clinic_id']}','{a['booked_on']}','{a['appt_date']}',{int(a['age'])},"
        f"'{a['gender']}',{a['distance_km']},{int(a['fee_inr'])},'{a['booking_channel']}',"
        f"'{a['visit_type']}','{a['reminder_sent']}',{int(a['prior_no_shows'])},"
        f"'{a['status']}',{int(a['lead_days'])},{int(a['no_show'])});"
    )

LOAD_SQL = os.path.join(ROOT, 'load_data.sql')
with open(LOAD_SQL, 'w') as f:
    f.write('\n'.join(lines) + '\n')

print('Wrote', LOAD_SQL)
print('Clinic INSERT statements:', len(clinics_df))
print('Appointment INSERT statements:', len(df))
print('Total INSERT statements:', len(clinics_df) + len(df))


appointment_id unique: True
Total missing values across the 16 columns: 0
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
Wrote /Users/harish/careloop-clinics-no-show-guard/appointments_clean.csv | rows: 990 | columns: 16
Wrote /Users/harish/careloop-clinics-no-show-guard/load_data.sql
Clinic INSERT statements: 8
Appointment INSERT statements: 990
Total INSERT statements: 998
